# 🏔️ Nepal Earthquake Damage Assessment
## Kavrepalanchok District Class Balance Analysis

### Project Overview
This notebook analyzes building damage patterns from the 2015 Nepal earthquake. The goal is to assess class balance in damage severity levels and create professional visualizations.

### Analysis Goals:
1. Load and explore the damage assessment data
2. Calculate class distribution statistics
3. Create professional visualizations
4. Assess class balance for machine learning readiness

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
from pathlib import Path

# Add project root to path for imports
sys.path.append('..')

# Import our modules
from src.visualization.simple_plot import (
    plot_damage_distribution,
    plot_damage_comparison,
    plot_feature_distribution,
    create_dashboard_figure
)

from config.settings import RAW_DATA_DIR, FIGURES_DIR, TARGET_COLUMN

print("✅ Libraries imported successfully")
print(f"📁 Data directory: {RAW_DATA_DIR}")
print(f"📁 Figures directory: {FIGURES_DIR}")

## 1. Load Data

In [ ]:
# Load the sample data
data_path = RAW_DATA_DIR / "sample_data.csv"

if data_path.exists():
    df = pd.read_csv(data_path)
    print(f"✅ Data loaded: {len(df):,} samples")
else:
    print("⚠️ Sample data not found. Generating new data...")
    from src.data.generate_sample_data import generate_earthquake_data, save_sample_data
    df = generate_earthquake_data()
    save_sample_data(df)

In [ ]:
# Display basic info
print("📊 Dataset Info:")
print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing Values:\n{df.isnull().sum()}")

In [ ]:
# First few rows
df.head()

## 2. Class Distribution Analysis

In [ ]:
# Calculate damage distribution
damage_counts = df[TARGET_COLUMN].value_counts().sort_index()
damage_props = df[TARGET_COLUMN].value_counts(normalize=True).sort_index()

# Create summary dataframe
dist_df = pd.DataFrame({
    'Count': damage_counts,
    'Proportion': damage_props
})

print("📈 Damage Distribution:")
dist_df

In [ ]:
# Calculate class balance metrics
n_classes = len(damage_counts)
max_class = damage_props.idxmax()
min_class = damage_props.idxmin()
max_prop = damage_props.max()
min_prop = damage_props.min()
imbalance_ratio = max_prop / min_prop

# Calculate entropy (measure of disorder)
entropy = -sum(p * np.log(p) for p in damage_props.values if p > 0)
max_entropy = np.log(n_classes)
normalized_entropy = entropy / max_entropy

# Calculate Gini coefficient
def gini_coefficient(x):
    """Calculate Gini coefficient"""
    x = np.array(x)
    mad = np.abs(np.subtract.outer(x, x)).mean()
    rmad = mad / np.mean(x)
    return 0.5 * rmad

gini = gini_coefficient(damage_props.values)

# Chi-square test for uniformity
expected = [len(df) / n_classes] * n_classes
chi2, p_value = stats.chisquare(damage_counts.values, expected)

print("⚖️  Class Balance Metrics:")
print("="*50)
print(f"Number of classes: {n_classes}")
print(f"Most frequent class: Level {max_class} ({max_prop:.2%})")
print(f"Least frequent class: Level {min_class} ({min_prop:.2%})")
print(f"Imbalance ratio (max/min): {imbalance_ratio:.2f}")
print(f"Entropy: {entropy:.3f} (max possible: {max_entropy:.3f})")
print(f"Normalized entropy: {normalized_entropy:.3f}")
print(f"Gini coefficient: {gini:.3f}")
print(f"Chi-square statistic: {chi2:.2f}")
print(f"P-value: {p_value:.4f}")

# Interpretation
print("\n📋 Interpretation:")
if p_value < 0.05:
    print("• Statistically significant deviation from uniform distribution")
else:
    print("• No significant deviation from uniform distribution")
    
if imbalance_ratio < 2:
    print("• Classes are very well balanced ✓")
elif imbalance_ratio < 4:
    print("• Classes are moderately balanced")
elif imbalance_ratio < 10:
    print("• Classes are imbalanced ⚠️")
else:
    print("• Classes are severely imbalanced 🔴")

if gini < 0.3:
    print("• Low inequality in class distribution")
elif gini < 0.5:
    print("• Moderate inequality in class distribution")
else:
    print("• High inequality in class distribution")

## 3. Basic Visualization

In [ ]:
# Create simple bar plot
fig, ax = plot_damage_distribution(df, save=True)
plt.show()

## 4. Comprehensive Dashboard

In [ ]:
# Create dashboard with multiple visualizations
fig = create_dashboard_figure(df, save=True)
plt.show()

## 5. Feature Analysis

In [ ]:
# Analyze building age distribution across damage levels
if 'building_age' in df.columns:
    fig, ax = plot_feature_distribution(df, 'building_age', save=True)
    plt.show()

In [ ]:
# Compare distribution by material type
if 'material_type' in df.columns:
    fig, axes = plot_damage_comparison(df, 'material_type', save=True)
    plt.show()

## 6. Correlation Analysis

In [ ]:
# Select numerical columns for correlation
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Calculate correlation matrix
corr_matrix = df[numerical_cols].corr()

# Plot heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlations', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. Conclusions and Recommendations

In [ ]:
print("="*70)
print("EXECUTIVE SUMMARY")
print("="*70)

print(f"""
📊 DATA QUALITY ASSESSMENT:
• Total samples analyzed: {len(df):,}
• Number of damage classes: {len(damage_counts)}
• Missing data: {df.isnull().sum().sum()} cells

⚖️ CLASS BALANCE ANALYSIS:
• Most frequent: Level {max_class} ({max_prop:.1%})
• Least frequent: Level {min_class} ({min_prop:.1%})
• Imbalance ratio: {imbalance_ratio:.2f}:1
• Gini coefficient: {gini:.3f}
• Statistical significance: p = {p_value:.4f}

💡 KEY INSIGHTS:
""")

if imbalance_ratio > 4:
    print("• Significant class imbalance detected - minority classes need attention")
    print("• Model performance may be biased toward majority classes")
else:
    print("• Classes are relatively balanced - suitable for standard modeling")

if p_value < 0.05:
    print("• Distribution is statistically different from uniform")
else:
    print("• Distribution approximates uniform distribution")

print(f"""
📋 RECOMMENDATIONS:
{'1. Use SMOTE or ADASYN for oversampling minority classes' if imbalance_ratio > 4 else '1. Standard modeling approaches are suitable'}
2. Collect additional data for underrepresented severity levels
3. Consider weighted loss functions in machine learning models
4. Implement stratified cross-validation

🎯 BUSINESS IMPACT:
• Priority areas: Focus data collection on levels {min_class-2}-{min_class+2}
• Resource allocation: Target {max_class}% of resources to most common damage types
• Risk assessment: Highest uncertainty for minority classes
""")

print("="*70)

## 8. Save Results

In [ ]:
# Save summary statistics to file
summary = {
    'total_samples': len(df),
    'n_classes': n_classes,
    'most_frequent_class': int(max_class),
    'most_frequent_proportion': float(max_prop),
    'least_frequent_class': int(min_class),
    'least_frequent_proportion': float(min_prop),
    'imbalance_ratio': float(imbalance_ratio),
    'entropy': float(entropy),
    'gini_coefficient': float(gini),
    'chi_square_p_value': float(p_value)
}

# Save to CSV
summary_df = pd.DataFrame([summary])
summary_path = REPORTS_DIR / 'analysis_summary.csv'
summary_df.to_csv(summary_path, index=False)
print(f"✅ Summary saved to: {summary_path}")

# Save class distribution
dist_df.to_csv(REPORTS_DIR / 'class_distribution.csv')
print(f"✅ Distribution saved to: {REPORTS_DIR / 'class_distribution.csv'}")

print("\n✅ Analysis complete!")